In [181]:
import math
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as Data

# 导读：Transformer 中译英（完整版）

---

## 通用 SOP：读任何项目/模型可以按这 5 步

| 步骤 | 做什么 | 产出 |
|------|--------|------|
| **1. 提取目标** | 从什么输入 → 什么输出 | 一句话：输入/输出是什么 |
| **2. 画数据流图** | 数据从哪来、经过谁、到哪去 | 一条线串到底的流程图 |
| **3. 画代码结构图** | 模块/类谁包含谁、谁调谁 | 树状或框图（如 Transformer → Encoder → …） |
| **4. 画张量形状简图** | 主要变量维度怎么变 | 表格或标注 (B, L, d) 等 |
| **5. 按执行顺序画代码地图/时序图** | 运行时先执行哪块、再执行哪块 | 顺序表或时序图 |

下面本 notebook 的导读就是按这 5 步组织的，换一个项目时同样可以按 1→2→3→4→5 做一遍。

---

**本模型一句话（SOP 第 1 步）**：中文句子 → 编码器得到「整句表示」→ 解码器根据该表示 + 已生成英文，逐词预测下一个词，直到结束符 E。

---

## 1. 数据流（SOP 第 2 步：一条线串到底）

```
sentences → make_data（补 padding）→ enc_inputs / dec_inputs / dec_outputs (LongTensor)
  → DataLoader 成 batch
  → Encoder: embedding → 位置编码 → N×[自注意力 + FFN] → enc_outputs
  → Decoder: embedding → 位置编码 → N×[自注意力 + 跨注意力 + FFN] → dec_outputs
  → projection → 词表 logits → CrossEntropyLoss(dec_outputs)
```

---

## 2. 代码结构图（SOP 第 3 步：谁包含谁）

```
Transformer
├── Encoder
│   ├── src_emb (Embedding)
│   ├── pos_emb (PositionalEncoding)
│   └── layers: [ Encoderlayer × n_layers ]
│         ├── enc_self_attn (MultiHeadAttention)
│         │     ├── W_Q, W_K, W_V, fc (Linear)
│         │     └── ScaledDotProductAttention  ← 单头注意力核心
│         └── pos_ffn (PoswiseFeedForwardNet: d_model→d_ff→d_model)
├── Decoder
│   ├── tgt_emb (Embedding)
│   ├── pos_emb (PositionalEncoding)
│   └── layers: [ DecoderLayer × n_layers ]
│         ├── dec_self_attn (MultiHeadAttention)   ← 只看已生成的解码端
│         ├── dec_enc_attn (MultiHeadAttention)   ← 看编码器输出
│         └── pos_ffn (PoswiseFeedForwardNet)
└── projection (Linear → 词表 logits)
```

**画图建议**：想搞清「这段代码在调谁」时，画这种**代码/模块结构图**就够用。

---

## 3. 张量形状简图（SOP 第 4 步：维度怎么变）

| 变量 | 形状（batch, 序列长, 特征） | 说明 |
|------|----------------------------|------|
| enc_inputs / dec_inputs | (B, L) | 词 id，B=batch_size，L=序列长 |
| src_emb(enc_inputs) | (B, L, d_model) | 每个 token 变成 d_model 维向量 |
| pos_emb 后 | (B, L, d_model) | 加位置编码，形状不变 |
| 进入 MultiHeadAttention 的 Q/K/V | (B, n_heads, L, d_k 或 d_v) | 多头时拆成 n_heads 份 |
| 单头 ScaledDotProduct: scores | (B, n_heads, L, L) | 每个位置对每个位置的相似度 |
| 单头 context | (B, n_heads, L, d_v) | 加权后的 V，再拼回 (B, L, d_model) |
| enc_outputs / dec_outputs | (B, L, d_model) | 编码器/解码器每层输出 |
| dec_logits (projection 后) | (B*L, tgt_vocab_size) | 每个位置预测词表上的分布 |

**画图建议**：卡在「这里维度对不上」「这里为什么要 view/reshape」时，画**矩阵/张量形状图**（谁 (B,L,d) 谁 (B,L,L)）最管用。

---

## 4. 代码地图（SOP 第 5 步：按执行顺序）

| 顺序 | 代码块 | 在干嘛 |
|------|--------|--------|
| 1 | sentences + src_vocab / tgt_vocab | 3 句中译英 + 源/目标词表（token→id） |
| 2 | make_data(sentences) | 句子→id 序列，**按最长句 padding**，得到 3 个 LongTensor |
| 3 | MyDataSet + DataLoader | 按 batch 打包，训练时 for 循环取 (enc, dec_in, dec_out) |
| 4 | d_model, d_ff, n_heads, device… | 超参 + **device（有 GPU 用 CUDA，否则 CPU）** |
| 5 | PositionalEncoding | 位置编码表 + dropout，加在 embedding 后 |
| 6 | get_attn_pad_mask / get_attn_subsequence_mask | padding 掩码 + 因果掩码（解码器不能看未来） |
| 7 | ScaledDotProductAttention | QK^T/√d_k → mask → softmax → 乘 V，**return context, attn** |
| 8 | MultiHeadAttention | 多组 Q/K/V，每组调 ScaledDotProduct，拼起来再 Linear + 残差 + LayerNorm |
| 9 | PoswiseFeedForwardNet | d_model→d_ff→d_model + 残差 + LayerNorm |
| 10 | Encoderlayer / DecoderLayer | 编码层：自注意力+FFN；解码层：自注意力+**跨注意力**+FFN |
| 11 | Encoder / Decoder | 堆 n_layers 层，Encoder 只算 enc_outputs；Decoder 用 enc_outputs 做 K/V |
| 12 | Transformer | Encoder + Decoder + projection，forward 返回 logits 和各类 attn（可选看） |
| 13 | model / criterion / optimizer | 模型 to(device)，CrossEntropyLoss(ignore_index=0)，SGD |
| 14 | 训练循环 | batch.to(device) → model(enc, dec) → loss → backward → step |

---

## 5. 建议阅读顺序

1. **先跑通**：从头执行到底，确认 loss 会降。
2. **抓数据线**：sentences → make_data（注意 padding）→ loader → model(enc_inputs, dec_inputs)。
3. **看结构**：对照上面的「代码结构图」，在 notebook 里找到 Transformer → Encoder → Encoderlayer → MultiHeadAttention → ScaledDotProductAttention。
4. **看形状**：在关键处 print(x.shape)，对照「张量形状简图」确认 (B,L,d_model) 等。
5. **抠细节**：ScaledDotProduct 的 QK^T、mask、softmax、乘 V；Decoder 里 dec_self_attn 与 dec_enc_attn 各看什么。

**两种图各有用**：**代码结构图** = 理清「谁调谁」；**矩阵/张量图** = 理清「维度怎么变」。按需画一种或两种，不必一次画全。

S 解码器输入
E 编码器输出
P 占位符（为什么要有这个？）
编码器输入要按「空格分词」后的 token 和 src_vocab 一致，这里按字分开写

微妙的P

In [182]:

sentences= [['我 是 教 师 P','S I am a teacher' ,'I am a teacher E'],
            ['我 喜 欢 教 学','S I like teaching P','I like teaching P E'],
            ['我 是 厨 师 P','S I am a cook' ,'I am a cook E']]

# 输入词汇表：key 必须和上面 .split() 得到的 token 一致（不能有多余空格）
src_vocab= {'P':0, '我':1, '是':2, '教':3, '师':4, '喜':5, '欢':6, '学':7, '厨':8}
src_idx2word = {src_vocab[key]: key for key in src_vocab}
src_vocab_size = len(src_vocab)

In [183]:
# 目标词汇表
tgt_vocab = {'P':0,'S':1,'E':2,'I':3,'am':4,'a':5,'teacher':6,'like':7,'teaching':8,'cook':9}
idx2word = {tgt_vocab[key]: key for key in tgt_vocab}
tgt_vocab_size = len(tgt_vocab)

In [184]:
src_len = len(sentences[0][0].split(" "))
print(src_len)
tgt_len = len(sentences[0][1].split(" "))
print(tgt_len)

5
5


义 make_data(）方法，将 sentences 转化为字典索引定义代码

In [185]:
def make_data(sentences):
    enc_inputs, dec_inputs, dec_outputs = [], [], []
    for i in range(len(sentences)):
        # 原始输入
        enc_input = [[src_vocab[n] for n in sentences[i][0].split()]]
        # 解码器输入
        dec_input = [[tgt_vocab[n] for n in sentences[i][1].split()]]
        # 解码器输出
        dec_output = [[tgt_vocab[n] for n in sentences[i][2].split()]]
        # 将原始输入、解码器输入、解码器输出添加到列表中
        enc_inputs.extend(enc_input)
        dec_inputs.extend(dec_input)
        dec_outputs.extend(dec_output)
    return torch.LongTensor(enc_inputs), torch.LongTensor(dec_inputs), torch.LongTensor(dec_outputs)

# 调用函数：把 sentences 转成三个 LongTensor，供后面模型用
enc_inputs, dec_inputs, dec_outputs = make_data(sentences)

print("enc_inputs:\n",enc_inputs)
print("\n\ndec_inputs:\n",dec_inputs)
print("\n\ndec_outputs:\n",dec_outputs)

enc_inputs:
 tensor([[1, 2, 3, 4, 0],
        [1, 5, 6, 3, 7],
        [1, 2, 8, 4, 0]])


dec_inputs:
 tensor([[1, 3, 4, 5, 6],
        [1, 3, 7, 8, 0],
        [1, 3, 4, 5, 9]])


dec_outputs:
 tensor([[3, 4, 5, 6, 2],
        [3, 7, 8, 0, 2],
        [3, 4, 5, 9, 2]])


In [186]:
class MyDataSet(Data.Dataset):
    def __init__(self, enc_inputs, dec_inputs, dec_outputs):
        super(MyDataSet, self).__init__()
        self.enc_inputs = enc_inputs
        self.dec_inputs = dec_inputs
        self.dec_outputs = dec_outputs
    def __len__(self):
        return self.enc_inputs.shape[0]
    def __getitem__(self, idx):
        return self.enc_inputs[idx], self.dec_inputs[idx], self.dec_outputs[idx]

loader = Data.DataLoader(MyDataSet(enc_inputs, dec_inputs, dec_outputs), 2, True)

print(loader.batch_size)

2


d_model嵌入的维度，是不是指一个token的文字转成向量的特征向量的维度
d_ff前向传播隐层维度是指前向传播时升维后的矩阵维度

In [187]:
d_model = 512 # 嵌入的维度 
d_ff = 2048 # 前向传播隐层维度
d_k = d_v = 64 # K 、 V 矩阵的维度
n_layers = 1 # 编码器和解码器的数俄
n_heads = 8 # 多头自汴，意力数

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # 无 GPU 时用 CPU


In [188]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout= 0.1, max_len = 5000):
        super(PositionalEncoding, self).__init__()
        self.dropout= nn.Dropout(p = dropout)
        pos_table = np.array([
        [pos / np.power(10000, 2 * i / d_model) for i in range(d_model)]
        if pos != 0 else np.zeros(d_model) for pos in range(max_len)])
        pos_table[1:, 0::2] = np.sin(pos_table[1:, 0::2])
        pos_table[1:, 1::2] = np.cos(pos_table[1:, 1::2])
        self.pos_table = torch.FloatTensor(pos_table).to(device)
    def forward(self, enc_inputs):
        enc_inputs += self.pos_table[:enc_inputs.size(1), :]
        return self.dropout(enc_inputs.to(device))

In [189]:
def get_attn_pad_mask(seq_q, seq_k):
    batch_size, len_q = seq_q.size()
    batch_size, len_k = seq_k.size()
    pad_attn_mask = seq_k.data.eq(0).unsqueeze(1)
    return pad_attn_mask.expand(batch_size, len_q, len_k)

def get_attn_subsequence_mask(seq):
    attn_shape = [seq.size(0), seq.size(1), seq.size(1)]
    subsequence_mask = np.triu(np.ones(attn_shape), k = 1)
    subsequence_mask = torch.from_numpy(subsequence_mask).byte()
    return subsequence_mask

In [ ]:
# 缩放点积注意力
# TODO 不太懂的东西
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()
    def forward(self, Q, K, V, attn_mask):
        scores = torch.matmul(Q, K.transpose(-1, -2)) / np.sqrt(d_k)
        scores.masked_fill_(attn_mask, -1e9)
        attn = nn.Softmax(dim=-1)(scores)
        context = torch.matmul(attn, V)
        return context, attn

In [ ]:
# 多头注意力
class MultiHeadAttention(nn.Module):
    def __init__(self):
        super(MultiHeadAttention, self).__init__()

        # 初始化三个线性层，用于将输入转换为查询、键和值
        # 没有太懂QKV的模式
        self.W_Q = nn.Linear(d_model, d_k * n_heads, bias= False)
        self.W_K = nn.Linear(d_model, d_k * n_heads, bias= False)
        self.W_V = nn.Linear(d_model, d_v * n_heads, bias= False)
        self.fc = nn.Linear(n_heads * d_v, d_model, bias= False)
    
    def forward(self, input_Q, input_K, input_V, attn_mask):
        residual, batch_size = input_Q, input_Q.size(0)
        # 先执行了运算，然后reshape，然后transpose
        Q = self.W_Q(input_Q).view(batch_size, -1, n_heads, d_k).transpose(1,2)
        K = self.W_K(input_K).view(batch_size, -1, n_heads, d_k).transpose(1,2)
        V = self.W_V(input_V).view(batch_size, -1, n_heads, d_v).transpose(1,2)
        attn_mask = attn_mask.unsqueeze(1).repeat(1, n_heads, 1, 1)
        context, attn= ScaledDotProductAttention()(Q, K, V, attn_mask)

        # 把注意力矩阵的维度从 (batch_size, n_heads, L, d_v) 转换为 (batch_size, L, n_heads, d_v)
        # 矩阵的塑性，又是受矩阵限制了。整个模型计算过程，矩阵塑形还蛮重要的
        context= context.transpose(1, 2).reshape(batch_size, -1, n_heads * d_v)

        # 全连接层，将输入从 n_heads * d_v 维处理到 d_model 维。感觉是矩阵维度的限制
        output= self.fc(context)

        # 把残差又给加回去了，避免在逐层计算时梯度消失
        return nn.LayerNorm(d_model).to(device)(output + residual), attn

nn.Linear 每个输出都由所有输入线性组合得到。改变输出矩阵大小，可升维绛维。
nn.ReLU() 激活函数，x < 0 时 y = 0。不改变输出大小。

以前的神经网络就是由这些基础单元组合起来的啊。。好粗糙啊

In [ ]:
# 前馈神经网络
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self):
        super(PoswiseFeedForwardNet, self).__init__()
        self.fc = nn.Sequential(
        # 全连接层，将输入从 d_model 维升到 d_ff 维
        # d_model 512维，d_ff 2048维
        nn.Linear(d_model, d_ff, bias= False),
        nn.ReLU(),
        nn.Linear(d_ff, d_model, bias= False))
    def forward(self, inputs):
        # 将输入作为残差，避免在逐层计算时梯度消失
        residual= inputs
        output = self.fc(inputs)
        # LayerNorm 是什么？归一化。。
        return nn.LayerNorm(d_model).to(device)(output + residual)

为什么EncoderLayer只需要1个多头，DecoderLayer需要2个？

编码器只关心「源句内部」；解码器既要关心「已生成目标句内部」，又要关心「整句源句」，所以解码器多一个「看编码器」的注意力。

这里的数据流图是没有画得很清晰的，得再看


In [ ]:
# 编码器与解码器
class Encoderlayer(nn.Module):
    def __init__(self):
        super(Encoderlayer, self).__init__()
        self.enc_self_attn = MultiHeadAttention()
        self.pos_ffn = PoswiseFeedForwardNet()
    def forward(self, enc_inputs, enc_self_attn_mask):
        enc_outputs, attn= self.enc_self_attn(enc_inputs, enc_inputs, enc_inputs, enc_self_attn_mask)
        enc_outputs = self.pos_ffn(enc_outputs)
        return enc_outputs, attn

class DecoderLayer(nn.Module):
    def __init__(self):
        super(DecoderLayer, self).__init__()
        self.dec_self_attn = MultiHeadAttention()
        self.dec_enc_attn = MultiHeadAttention()
        self.pos_ffn = PoswiseFeedForwardNet()
    def forward(self, dec_inputs, enc_outputs, dec_self_attn_mask, dec_enc_attn_mask):
        dec_outputs, dec_self_attn = self.dec_self_attn(dec_inputs, dec_inputs, dec_inputs, dec_self_attn_mask)
        dec_outputs, dec_enc_attn = self.dec_enc_attn(dec_outputs, enc_outputs, enc_outputs, dec_enc_attn_mask)
        dec_outputs = self.pos_ffn(dec_outputs)
        return dec_outputs, dec_self_attn, dec_enc_attn

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super(Encoder, self).__init__()
        # 初始化输入的embedding矩阵，在 enc_outputs = self.src_emb(enc_inputs) 中真正使用
        self.src_emb = nn.Embedding(src_vocab_size, d_model)
        # 初始化位置编码
        self.pos_emb = PositionalEncoding(d_model)
        # 初始化编码器层
        self.layers= nn.ModuleList([Encoderlayer() for _ in range(n_layers)])
    def forward(self, enc_inputs):
        # 调用embedding层/src_emb的forward，得到embedding输出
        enc_outputs = self.src_emb(enc_inputs)
        # 调用位置编码层的forward，加上位置编码输出
        enc_outputs = self.pos_emb(enc_outputs)
        # 增加掩码，让模型忽略无谓的标注P
        enc_self_attn_mask = get_attn_pad_mask(enc_inputs, enc_inputs)
        enc_self_attns = []
        for layer in self.layers:
            # 逐层调用编码器
            enc_outputs, enc_self_attn = layer(enc_outputs, enc_self_attn_mask)
            # 记录每一层的注意力，每层一个矩阵
            enc_self_attns.append(enc_self_attn)
        # 返回编码器输出和每层的注意力
        # 这里的enc_self_attns在后文没有参与计算
        return enc_outputs, enc_self_attns

In [ ]:
class Decoder(nn.Module):
    def __init__(self):
        super(Decoder, self).__init__()
        self.tgt_emb = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_emb = PositionalEncoding(d_model)
        self.layers= nn.ModuleList([DecoderLayer() for _ in range(n_layers)])

    def forward(self, dec_inputs, enc_inputs, enc_outputs):
        dec_outputs = self.tgt_emb(dec_inputs)
        dec_outputs = self.pos_emb(dec_outputs)

        # 这些都是在实现「解码器该怎么看、不该看什么
        #       变量	                            作用（工程上要满足的约束）
        # dec_self_attn_pad_mask	        解码器自注意力里：不要注意 decoder 序列中的 PAD（和 Encoder 的 pad mask 同理）。
        # dec_self_attn_subsequence_mask	解码器自注意力里：不能看“未来”位置（因果 / 因果掩码）。预测第 t 个词时，只能用 1..t，不能用到 t+1, t+2...。
        # dec_self_attn_mask	            把上面两个合并：PAD 不能看 + 未来不能看，解码器自注意力只用这一份 mask。
        # dec_enc_attn_mask	                解码器看编码器时：不要注意 encoder 序列里的 PAD（encoder 那边补的 PAD 不参与跨注意力）。

        dec_self_attn_pad_mask = get_attn_pad_mask(dec_inputs, dec_inputs).to(device)
        dec_self_attn_subsequence_mask = get_attn_subsequence_mask(dec_inputs).to(device)
        dec_self_attn_mask = torch.gt((dec_self_attn_pad_mask + dec_self_attn_subsequence_mask), 0).to(device)
        dec_enc_attn_mask = get_attn_pad_mask(dec_inputs, enc_inputs)

        dec_self_attns, dec_enc_attns = [],[]
        for layer in self.layers:
            dec_outputs, dec_self_attn, dec_enc_attn = layer(dec_outputs, enc_outputs, dec_self_attn_mask, dec_enc_attn_mask)
            dec_self_attns.append(dec_self_attn)
            dec_enc_attns.append(dec_enc_attn)
        return dec_outputs, dec_self_attns, dec_enc_attns

model(enc_inputs, dec_inputs)
    │
    ▼
Module.__call__   (1912 行：其实就是 _wrapped_call_impl 的引用)
    │
    ▼
_wrapped_call_impl (1769 行)
    │
    ├─ 若 _compiled_call_impl 不是 None（你用了 torch.compile(model)）
    │      → 用编译后的版本 _compiled_call_impl(*args, **kwargs)
    │
    └─ 否则（你没 compile，一般情况）
           → _call_impl(*args, **kwargs)
                │
                ▼
            _call_impl (1776 行)
                │
                ├─ 没有各种 hook 时（你一般也没注册）
                │      → 直接 forward_call(*args, **kwargs)
                │         而 forward_call = self.forward（1777 行）
                │      → 也就是你的 Transformer.forward(enc_inputs, dec_inputs)
                │
                └─ 有 forward/backward hook 时
                       → 先跑 pre_hook，再 forward_call(...)，再 post_hook

enc_outputs 编码器输出
enc_self_attns 编码器的每层注意力

dec_outputs 解码器输出
dec_self_attns 解码器的每层注意力
dec_enc_attns ？

In [ ]:
class Transformer(nn.Module):
    def __init__(self):
        super(Transformer, self).__init__()
        self.Encoder = Encoder().to(device)
        self.Decoder = Decoder().to(device)
        self.projection = nn.Linear(d_model, tgt_vocab_size, bias=False).to(device)
    def forward(self, enc_inputs, dec_inputs):
        # __call__ → _wrapped_call_impl → _call_impl → self.forward(...)（即 Transformer.forward）
        # 经过编码器后，返回的输出和每层的注意力
        enc_outputs, enc_self_attns = self.Encoder(enc_inputs)
        # print("\n\nenc_outputs:\n",enc_outputs)
        # print("\n\nattention:\n",enc_self_attns)
        # 经过解码器
        dec_outputs, dec_self_attns, dec_enc_attns = self.Decoder(dec_inputs, enc_inputs, enc_outputs)
        # 这一层是干嘛的？
        # 通常是 nn.Linear(d_model, tgt_vocab_size)，把 d_model 维 → 词表大小维。
        dec_logits = self.projection(dec_outputs)
        return dec_logits.view(-1, dec_logits.size(-1)), enc_self_attns, dec_self_attns, dec_enc_attns

In [197]:
model = Transformer().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.SGD(model.parameters(), lr = 1e-3, momentum = 0.99)

In [198]:
for epoch in range(50):
    for enc_inputs, dec_inputs, dec_outputs in loader:
        # 把三个数据放在同一设备上
        enc_inputs, dec_inputs, dec_outputs = enc_inputs.to(device), dec_inputs.to(device), dec_outputs.to(device)
        outputs, enc_self_attns, dec_self_attns, dec_enc_attns = model(enc_inputs,dec_inputs)
        loss = criterion(outputs, dec_outputs.view(-1))
        print('Epoch:','%04d'% (epoch+ 1),'loss=','{:.6f}'.format(loss))
        optimizer.zero_grad()
        loss. backward ()
        optimizer. step ()

Epoch: 0001 loss= 2.425428
Epoch: 0001 loss= 2.216399
Epoch: 0002 loss= 2.212177
Epoch: 0002 loss= 2.144006
Epoch: 0003 loss= 1.877402
Epoch: 0003 loss= 1.626046
Epoch: 0004 loss= 1.407251
Epoch: 0004 loss= 1.158261
Epoch: 0005 loss= 1.030348
Epoch: 0005 loss= 0.746033
Epoch: 0006 loss= 0.863853
Epoch: 0006 loss= 0.474810
Epoch: 0007 loss= 0.644223
Epoch: 0007 loss= 0.271137
Epoch: 0008 loss= 0.532008
Epoch: 0008 loss= 0.303673
Epoch: 0009 loss= 0.530523
Epoch: 0009 loss= 0.189397
Epoch: 0010 loss= 0.514347
Epoch: 0010 loss= 0.198506
Epoch: 0011 loss= 0.434592
Epoch: 0011 loss= 0.091713
Epoch: 0012 loss= 0.332243
Epoch: 0012 loss= 0.225407
Epoch: 0013 loss= 0.162251
Epoch: 0013 loss= 0.303399
Epoch: 0014 loss= 0.200560
Epoch: 0014 loss= 0.150014
Epoch: 0015 loss= 0.145130
Epoch: 0015 loss= 0.168600
Epoch: 0016 loss= 0.190015
Epoch: 0016 loss= 0.324481
Epoch: 0017 loss= 0.353714
Epoch: 0017 loss= 0.025418
Epoch: 0018 loss= 0.140019
Epoch: 0018 loss= 0.374484
Epoch: 0019 loss= 0.158208
E

In [207]:
def test(model, enc_input, start_symbol):
    enc_outputs, enc_self_attns = model.Encoder (enc_input)
    dec_input = torch.zeros(1, tgt_len).type_as(enc_input.data)
    next_symbol = start_symbol
    for i in range(0, tgt_len):
        dec_input[0][i] = next_symbol
        dec_outputs, _, _ = model.Decoder(dec_input, enc_input, enc_outputs)
        projected = model.projection(dec_outputs)
        prob= projected.squeeze(0).max(dim = -1, keepdim = False)[1]
        next_word = prob.data[i]
        next_symbol = next_word. item()
    return dec_input
enc_inputs, _, _ = next(iter(loader))
predict_dec_input = test(model, enc_inputs[0].view(1, -1).to(device), start_symbol = tgt_vocab["S"])
predict, _, _, _ = model(enc_inputs [0]. view(1, -1).to(device), predict_dec_input)
predict= predict.data.max(1, keepdim = True)[1]
print([src_idx2word[int(i)] for i in enc_inputs[0]],' ->', [idx2word[n.item()] for n in predict.squeeze()])

['我', '是', '厨', '师', 'P']  -> ['I', 'am', 'a', 'cook', 'E']
